# B03a: Amplitude Envelope Extraction (Quick Version)

**Duration**: ~25 minutes  
**Prerequisites**: B01, A04b

> 💡 **Quick Version**: Imports pre-built functions from `src/envelope.py`. See the full tutorial for implementation details.


## Table of Contents

1. [Introduction](#section-1-introduction)
2. [Extraction Pipeline](#section-2-extraction)
3. [Temporal Dynamics](#section-3-dynamics)
4. [Smoothing Methods](#section-4-smoothing)
5. [Exercises](#section-5-exercises)
6. [Summary](#summary)
7. [External Resources](#external-resources)
8. [Discussion Questions](#discussion-questions)


In [ ]:
# =============================================================================
# Imports
# =============================================================================

# Standard library
import sys
from pathlib import Path

# Third-party
import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# Local imports
src_path = Path.cwd().parent.parent.parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from envelope import (
    extract_envelope,
    compute_envelope_psd,
    smooth_envelope_moving_average,
    smooth_envelope_gaussian,
    smooth_envelope_lowpass,
)
from filtering import bandpass_filter
from colors import COLORS

# Matplotlib configuration
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11

np.random.seed(42)
print("Imports loaded ✓")


---

<a id="section-1-introduction"></a>
## 1. Introduction

The **amplitude envelope** traces the slow-varying amplitude modulation of an oscillation.

**Pipeline**: Bandpass filter → Hilbert transform → Magnitude

$$A(t) = |x_{\text{filtered}}(t) + i \cdot \mathcal{H}\{x_{\text{filtered}}(t)\}|$$

Key insight: the envelope operates on **timescales much slower** than the carrier frequency.
Typical alpha (10 Hz) envelopes fluctuate at < 2 Hz.


In [ ]:
# Demonstrate the envelope concept on a simple AM signal
fs = 500
t = np.arange(0, 3.0, 1/fs)

# Carrier with slow amplitude modulation
carrier = np.sin(2 * np.pi * 10 * t)
modulation = 0.5 + 0.5 * np.sin(2 * np.pi * 0.5 * t)
am_signal = modulation * carrier

# Extract envelope using Hilbert transform directly
analytic = hilbert(am_signal)
envelope = np.abs(analytic)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, am_signal, color=COLORS["signal_1"], lw=0.8, alpha=0.7, label="AM signal (10 Hz)")
ax.plot(t, envelope, color=COLORS["negative"], lw=2.5, label="Envelope")
ax.plot(t, -envelope, color=COLORS["negative"], lw=2.5)
ax.plot(t, modulation, color=COLORS["signal_3"], lw=2, ls="--", label="True modulation")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Amplitude Envelope — Hilbert Transform", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("→ The envelope (red) tracks the slow amplitude modulation (green dashed)")


---

<a id="section-2-extraction"></a>
## 2. Extraction Pipeline

`extract_envelope(signal, fs, band)` applies the full pipeline:
1. Bandpass filter to isolate the frequency band
2. Hilbert transform → analytic signal
3. Take the magnitude → envelope


In [ ]:
# EEG-like signal with multiple frequency components
np.random.seed(42)
fs_eeg = 250
t_eeg = np.arange(0, 5.0, 1/fs_eeg)

# Alpha burst with varying amplitude
alpha_mod = 0.2 + np.exp(-((t_eeg - 1.5) ** 2) / 0.1) + 0.8 * np.exp(-((t_eeg - 3.5) ** 2) / 0.15)
alpha = alpha_mod * np.sin(2 * np.pi * 10 * t_eeg)
noise = 0.3 * np.random.randn(len(t_eeg))
eeg = alpha + 0.5 * np.sin(2 * np.pi * 2 * t_eeg) + noise  # add delta + noise

# Extract alpha envelope
alpha_envelope = extract_envelope(eeg, fs_eeg, band=(8, 13))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(t_eeg, eeg, color=COLORS["signal_1"], lw=0.7, alpha=0.8, label="Raw EEG")
axes[0].set_ylabel("Amplitude (a.u.)")
axes[0].set_title("EEG-like Signal with Alpha Bursts", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_eeg, alpha_envelope, color=COLORS["negative"], lw=2, label="Alpha envelope (8–13 Hz)")
axes[1].fill_between(t_eeg, 0, alpha_envelope, alpha=0.3, color=COLORS["negative"])
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Amplitude (a.u.)")
axes[1].set_title("Extracted Alpha Amplitude Envelope", fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("→ The envelope reveals alpha burst timing and intensity")


---

<a id="section-3-dynamics"></a>
## 3. Temporal Dynamics

The envelope has its **own power spectrum** — this reveals *how fast* amplitude fluctuates.


In [ ]:
freqs_env, psd_env = compute_envelope_psd(alpha_envelope, fs_eeg)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(freqs_env, psd_env, color=COLORS["negative"], lw=2)
ax.set_xlim(0, 8)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power (a.u.²/Hz)")
ax.set_title("Power Spectrum of the Alpha Envelope", fontweight="bold")
ax.grid(True, alpha=0.3)
ax.axvline(2, color="gray", ls="--", alpha=0.5, label="Typical envelope bandwidth (<2 Hz)")
ax.legend()
plt.tight_layout()
plt.show()
print("→ Envelope fluctuations are concentrated at low frequencies (<2 Hz)")


---

<a id="section-4-smoothing"></a>
## 4. Smoothing Methods

Three smoothing methods with different trade-offs:

| Method | Trade-off |
|--------|-----------|
| Moving average | Simple, can introduce edge discontinuities |
| Gaussian | Smooth output, σ controls temporal spread |
| Low-pass filter | Principled frequency cutoff |


In [ ]:
sigma_samples = int(0.1 * fs_eeg)  # 100 ms

env_ma  = smooth_envelope_moving_average(alpha_envelope, window_samples=sigma_samples)
env_gau = smooth_envelope_gaussian(alpha_envelope, sigma_samples=sigma_samples)
env_lp  = smooth_envelope_lowpass(alpha_envelope, fs_eeg, cutoff_hz=2.0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t_eeg, alpha_envelope, color=COLORS["signal_1"], lw=0.8, alpha=0.5, label="Raw envelope")
ax.plot(t_eeg, env_ma,  color=COLORS["negative"],  lw=2,   label="Moving average")
ax.plot(t_eeg, env_gau, color=COLORS["signal_3"],  lw=2,   ls="--", label="Gaussian (σ=100 ms)")
ax.plot(t_eeg, env_lp,  color=COLORS["high_sync"], lw=2,   ls=":", label="Low-pass (2 Hz)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (a.u.)")
ax.set_title("Comparing Smoothing Methods", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---

<a id="section-5-exercises"></a>
## 5. Exercises

### 🎯 Exercise 1: Band Comparison

**Difficulty**: 🟢 Easy  
**Task**: Extract and plot envelopes for delta (1–4 Hz), alpha (8–13 Hz), and beta (13–30 Hz) bands from the same EEG signal. Compare their peak amplitudes.

```python
# Your code here
bands = {"delta": (1, 4), "alpha": (8, 13), "beta": (13, 30)}
band_colors = {"delta": COLORS["signal_1"], "alpha": COLORS["negative"], "beta": COLORS["signal_3"]}

# Extract each envelope and plot
# ...
```

### 🎯 Exercise 2: Smoothing Parameter Sensitivity

**Difficulty**: 🟡 Medium  
**Task**: Compute Gaussian-smoothed envelopes with σ = 25, 100, 250 ms. For each, compute the standard deviation of the envelope. What trend do you observe?

```python
# Your code here
sigmas_ms = [25, 100, 250]
for sigma_ms in sigmas_ms:
    sigma_samples = int(sigma_ms / 1000 * fs_eeg)
    env_smooth = smooth_envelope_gaussian(alpha_envelope, sigma_samples=sigma_samples)
    print(f"σ = {sigma_ms} ms → std = {env_smooth.std():.4f}")
```


---

<a id="summary"></a>
## 6. Summary

### Key Concepts

| Concept | Key Point |
|---------|-----------|
| **Amplitude envelope** | Magnitude of the analytic signal: `|hilbert(filtered_signal)|` |
| **Extraction pipeline** | Bandpass filter → Hilbert → Magnitude |
| **Envelope timescale** | Much slower than carrier (typically < 2 Hz for cognition) |
| **Smoothing** | Reduces noise, but trades off temporal resolution |

### Functions from `src/envelope.py`

| Function | Signature |
|----------|-----------|
| `extract_envelope` | `(signal, fs, band) → NDArray` |
| `compute_envelope_psd` | `(envelope, fs) → (freqs, psd)` |
| `smooth_envelope_gaussian` | `(envelope, sigma_samples) → NDArray` |
| `smooth_envelope_lowpass` | `(envelope, fs, cutoff_hz) → NDArray` |


---

<a id="external-resources"></a>
## 7. External Resources

### 📚 Scientific References

- **Gabor, D. (1946)** - *Theory of communication* — Analytic signals and instantaneous amplitude
- **Cohen, M.X. (2014)** - *Analyzing Neural Time Series Data* — Chapter 11: Hilbert transform

### 🎧 NotebookLM Resources

*(NotebookLM links will be added once this notebook is finalized)*


---

<a id="discussion-questions"></a>
## 8. Discussion Questions

1. **Why filter first?** What happens if you apply the Hilbert transform to a broadband EEG signal without filtering?

2. **Smoothing vs resolution**: You need to track 500 ms amplitude bursts. What smoothing σ would you choose? What is the minimum detectable burst duration?

3. **Amplitude vs Power**: When should you use the envelope (amplitude) vs the squared envelope (power)? Give one example where the distinction matters.
